# Saved Coverage and Eight-Contrast Diagnostics

Diagnose saved confidence intervals for the fixed initial-amount S1/random-retention cell, and compute four difference/sum contrast pairs from its first saved mixed-model fit. These are separate diagnostics, not a newly validated inference method.


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
if sys.version_info[:3] != (3, 10, 9):
    raise RuntimeError('Use Python 3.10.9 for this reference-matched workflow.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'numpy==1.23.5', 'scipy==1.10.0', 'pandas==2.0.3', 'matplotlib==3.7.5'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(ROOT)])


## Inputs and Scope

Use the existing public input producer below, or point `INPUT_ROOT` to its completed output. Preparation binds saved bytes only and fails if they are missing; it never silently generates or fits a model. Optional input production is separate and can be expensive.

Coverage needs the 2,000 target members, not all 80,000 grid members. The single-fit calculation needs ordinal 32 only. R 4.2.2, the [reference R packages](../../configs/conditional_statistics_environment.json), and GNU `sha256sum` are required for fit computation; Node.js 22 is additionally required for input production. No human data are used.


In [ ]:
import json
import shlex

INPUT_ROOT = ROOT / 'results' / 'saved_diagnostic_inputs'
OUTPUT = ROOT / 'results' / 'saved_diagnostics'
RSCRIPT = 'Rscript'
NODE = 'node'
RUN_INPUT_PRODUCER = False
INCLUDE_COVERAGE_INPUTS = False
RUN_COVERAGE = False
RUN_SAVED_FIT = False
# For an existing output with multiple aggregates, set this explicitly.
COVERAGE_AGGREGATE = None

def command(args, run=False):
    args = [str(arg) for arg in args]
    print(shlex.join(args))
    if run:
        OUTPUT.mkdir(parents=True, exist_ok=True)
        subprocess.run(args, cwd=ROOT, check=True)


## Optional Input Production

This calls the existing fixed-seed producer. It creates a **new public reproduction**, not the historical archived files. Leave disabled when reusing saved inputs. Set `INCLUDE_COVERAGE_INPUTS=True` only when producing the full 2,000-member diagnostic roster. Unavailable fits and adverse results are retained by the original producer.


In [ ]:
ordinals = [40 * (rep - 1) + 32 for rep in range(1, 2001)] if INCLUDE_COVERAGE_INPUTS else [32]
command([sys.executable, 'scripts/reproduce_conditional_statistics.py', 'run', '--execute',
         '--family', 'power', '--ids', ','.join(map(str, ordinals)), '--batch-size', '20',
         '--output', INPUT_ROOT, '--rscript', RSCRIPT, '--node', NODE], RUN_INPUT_PRODUCER)


## Saved Coverage Diagnosis

The producer and independent reader reopen the original saved interval endpoints, retain the exact target roster and recompute coverage, miss sides, paired misses, standardized errors and retained-count diagnostics. The historical accepted result was CR2 1,879/2,000 (0.9395), versus baseline 1,881/2,000 (0.9405), with 109 common misses. This remains an applicability warning, not an identified bug or a correction. New-public outcomes are not forced to equal those counts.


In [ ]:
aggregate = Path(COVERAGE_AGGREGATE) if COVERAGE_AGGREGATE else None
if aggregate is None:
    candidates = sorted((INPUT_ROOT / 'power').glob('aggregate*.json'))
    if len(candidates) == 1:
        aggregate = candidates[0]
    elif RUN_COVERAGE:
        raise ValueError('Set COVERAGE_AGGREGATE to the target-complete aggregate printed by the input producer.')
if aggregate is not None:
    command([sys.executable, 'scripts/diagnose_saved_coverage.py', 'prepare-public',
             '--aggregate', aggregate, '--events-directory', INPUT_ROOT / 'power' / 'events',
             '--output', OUTPUT / 'coverage_input.json'], RUN_COVERAGE)
    command([sys.executable, 'scripts/diagnose_saved_coverage.py', 'diagnose',
             '--request', OUTPUT / 'coverage_input.json', '--output', OUTPUT / 'coverage.json'], RUN_COVERAGE)
    command([sys.executable, 'scripts/read_saved_coverage.py',
             '--request', OUTPUT / 'coverage_input.json', '--producer-result', OUTPUT / 'coverage.json',
             '--output', OUTPUT / 'coverage_independent.json'], RUN_COVERAGE)
else:
    print('Coverage inputs not selected. Generate or select the complete saved target roster first.')


## Single Saved-Fit Eight-Contrast Diagnostic

For the fixed target's first fit, calculate all eight named contrasts with their own variance and degrees of freedom. The independent reader derives them from the saved model using lmerTest and clubSandwich. It also verifies the actual request/frame, cluster counts and source identity. The historical object had 149/156/147 retained participants; the adapters use actual counts, not fixed nominal counts.

Agreement on one object does not establish population calibration, magnitude-test power, a primary human-analysis method, or the adequacy of a recruitment target.


In [ ]:
command([sys.executable, 'scripts/reproduce_saved_fit_magnitude.py', 'prepare',
         '--reproduction-root', INPUT_ROOT, '--output', OUTPUT / 'fit_input.json'], RUN_SAVED_FIT)
command([sys.executable, 'scripts/reproduce_saved_fit_magnitude.py', 'compute',
         '--input', OUTPUT / 'fit_input.json', '--rscript', RSCRIPT,
         '--output', OUTPUT / 'eight_contrasts.json'], RUN_SAVED_FIT)
command([sys.executable, 'scripts/reproduce_saved_fit_magnitude.py', 'verify',
         '--input', OUTPUT / 'fit_input.json', '--producer-result', OUTPUT / 'eight_contrasts.json',
         '--rscript', RSCRIPT, '--output', OUTPUT / 'eight_contrasts_independent.json'], RUN_SAVED_FIT)


## Inspect Outputs

Keep each result with its input binding and independent proof. Outputs use new-file semantics: use a fresh output destination for another check. Historical archives can instead be admitted through the explicit historical modes in the [CLI guide](../../scripts/README.md); their descriptors are never relabeled as new public production.


In [ ]:
for name in ('coverage.json', 'coverage_independent.json', 'eight_contrasts.json', 'eight_contrasts_independent.json'):
    path = OUTPUT / name
    if path.is_file():
        result = json.loads(path.read_text())
        print(name)
        print(json.dumps(result, indent=2)[:12000])
